# 00 — First principles of AI-powered data analysis

| | |
|---|---|
| **Track** | Applied Labs |
| **Domain** | AG — AI Data Analyst |
| **Level** | Advanced |
| **Time** | ~30 min |
| **Prerequisites** | Python, SQL basics |
| **Open in Colab** | [![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai2026/castalia/blob/main/labs/06_ai_data_analyst/00_first_principles.ipynb) |

**What you'll learn:**
- What text-to-SQL is and why it matters
- Why translating natural language to SQL is harder than it looks
- How self-correction loops fix broken queries
- Why narration transforms raw data into insight
- How schema understanding grounds generation
- The $30B+ market opportunity

## 1 · Setup

In [ ]:
!pip install -q "pandas>=2.0.0" "matplotlib>=3.7.0"

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional

print("✅ Setup complete")

## 2 · What is text-to-SQL?

Text-to-SQL translates **natural-language questions** into **executable SQL queries**,
grounded in a specific database schema. The LLM must understand:

1. The **question intent** — what the user actually wants to know
2. The **schema** — which tables, columns, and relationships exist
3. The **SQL dialect** — correct syntax for the target database
4. The **business context** — what "revenue" or "active user" means here

In [ ]:
# Five example question → SQL translations showing increasing complexity

EXAMPLE_PAIRS: list[dict[str, str]] = [
    {
        "question": "How many customers do we have?",
        "sql": "SELECT COUNT(*) AS customer_count FROM customers;",
        "complexity": "basic — single table, single aggregate"
    },
    {
        "question": "What are our top 5 products by revenue?",
        "sql": (
            "SELECT p.name, SUM(oi.quantity * oi.unit_price) AS revenue "
            "FROM order_items oi "
            "JOIN products p ON oi.product_id = p.id "
            "GROUP BY p.name ORDER BY revenue DESC LIMIT 5;"
        ),
        "complexity": "medium — JOIN, aggregate, ORDER BY, LIMIT"
    },
    {
        "question": "Which customers placed orders in every quarter of 2024?",
        "sql": (
            "SELECT c.name FROM customers c "
            "JOIN orders o ON c.id = o.customer_id "
            "WHERE strftime('%Y', o.order_date) = '2024' "
            "GROUP BY c.id, c.name "
            "HAVING COUNT(DISTINCT ((CAST(strftime('%m', o.order_date) AS INT) - 1) / 3 + 1)) = 4;"
        ),
        "complexity": "hard — date extraction, HAVING, division logic"
    },
    {
        "question": "What is the month-over-month revenue trend?",
        "sql": (
            "SELECT strftime('%Y-%m', o.order_date) AS month, "
            "SUM(oi.quantity * oi.unit_price) AS revenue "
            "FROM orders o JOIN order_items oi ON o.id = oi.order_id "
            "GROUP BY month ORDER BY month;"
        ),
        "complexity": "medium — date grouping, multi-table JOIN"
    },
    {
        "question": "Which product categories have declining sales over the last 3 months?",
        "sql": (
            "WITH monthly AS ("
            "  SELECT c.name AS category, strftime('%Y-%m', o.order_date) AS month, "
            "  SUM(oi.quantity * oi.unit_price) AS revenue "
            "  FROM order_items oi "
            "  JOIN products p ON oi.product_id = p.id "
            "  JOIN categories c ON p.category_id = c.id "
            "  JOIN orders o ON oi.order_id = o.id "
            "  GROUP BY c.name, month"
            ") SELECT * FROM monthly ORDER BY category, month;"
        ),
        "complexity": "hard — CTE, 4-table JOIN, trend analysis"
    }
]

for i, pair in enumerate(EXAMPLE_PAIRS, 1):
    print(f"{'─'*60}")
    print(f"Example {i} [{pair['complexity']}]")
    print(f"  Q: {pair['question']}")
    print(f"  SQL: {pair['sql'][:80]}{'...' if len(pair['sql']) > 80 else ''}")
print(f"{'─'*60}")

## 3 · Why it's harder than it looks

A single question can map to **multiple valid SQL queries** depending on
how you interpret ambiguous terms. The model must handle:

- **Lexical ambiguity** — "top" could mean by revenue, count, or recency
- **Scope ambiguity** — "last quarter" depends on current date
- **Join reasoning** — the model must infer which tables to connect
- **Business logic** — domain-specific definitions not in the schema

In [ ]:
def show_ambiguity(question: str, interpretations: list[dict[str, str]]) -> None:
    """Display multiple valid SQL interpretations for an ambiguous question."""
    print(f"Question: \"{question}\"\n")
    for i, interp in enumerate(interpretations, 1):
        print(f"  Interpretation {i}: {interp['assumption']}")
        print(f"  SQL: {interp['sql']}")
        print()


# Example 1: "top customers" is ambiguous
show_ambiguity(
    "Who are our top customers?",
    [
        {
            "assumption": "Top by total revenue (lifetime)",
            "sql": (
                "SELECT c.name, SUM(oi.quantity * oi.unit_price) AS total_revenue "
                "FROM customers c JOIN orders o ON c.id = o.customer_id "
                "JOIN order_items oi ON o.id = oi.order_id "
                "GROUP BY c.name ORDER BY total_revenue DESC LIMIT 10;"
            )
        },
        {
            "assumption": "Top by order frequency",
            "sql": (
                "SELECT c.name, COUNT(DISTINCT o.id) AS order_count "
                "FROM customers c JOIN orders o ON c.id = o.customer_id "
                "GROUP BY c.name ORDER BY order_count DESC LIMIT 10;"
            )
        },
        {
            "assumption": "Top by most recent activity",
            "sql": (
                "SELECT c.name, MAX(o.order_date) AS last_order "
                "FROM customers c JOIN orders o ON c.id = o.customer_id "
                "GROUP BY c.name ORDER BY last_order DESC LIMIT 10;"
            )
        }
    ]
)

In [ ]:
# Demonstrate JOIN reasoning complexity

JOIN_EXAMPLES: list[dict[str, str]] = [
    {
        "question": "What products did customer 'Alice' buy?",
        "tables_needed": ["customers", "orders", "order_items", "products"],
        "join_chain": "customers → orders → order_items → products",
        "reasoning": "Must traverse 3 joins to connect customer name to product name"
    },
    {
        "question": "Which categories are most popular in Europe?",
        "tables_needed": ["customers", "orders", "order_items", "products", "categories"],
        "join_chain": "customers (region filter) → orders → order_items → products → categories",
        "reasoning": "5-table join; region is on customers, category is on products"
    }
]

for ex in JOIN_EXAMPLES:
    print(f"Q: {ex['question']}")
    print(f"  Tables: {' → '.join(ex['tables_needed'])}")
    print(f"  Chain:  {ex['join_chain']}")
    print(f"  Why:    {ex['reasoning']}\n")

## 4 · The self-correction loop

Text-to-SQL systems improve dramatically when they can **detect and fix errors**:

```
Generate SQL → Execute → Check results → Error?
    ↑                                        ↓
    └──── Analyze error + Regenerate ←───────┘
```

Common failure modes:
- **Syntax errors** — missing quotes, wrong function names
- **Wrong JOINs** — empty results from incorrect join columns
- **Wrong aggregation** — grouping by the wrong column
- **Missing filters** — returning all-time instead of last quarter

In [ ]:
# Simulate a self-correction loop with a real SQLite database

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create a simple schema
cursor.executescript("""
    CREATE TABLE departments (id INTEGER PRIMARY KEY, name TEXT);
    CREATE TABLE employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        dept_id INTEGER REFERENCES departments(id),
        salary REAL
    );
    INSERT INTO departments VALUES (1, 'Engineering'), (2, 'Sales'), (3, 'Marketing');
    INSERT INTO employees VALUES (1, 'Alice', 1, 95000), (2, 'Bob', 1, 105000),
        (3, 'Carol', 2, 75000), (4, 'Dave', 2, 80000), (5, 'Eve', 3, 70000);
""")

def execute_sql(sql: str) -> tuple[bool, str]:
    """Execute SQL and return (success, result_or_error)."""
    try:
        df = pd.read_sql_query(sql, conn)
        if df.empty:
            return False, "Query returned empty results"
        return True, df.to_string(index=False)
    except Exception as e:
        return False, str(e)


# Attempt 1: Wrong join column (simulated error)
attempt_1 = "SELECT d.name, AVG(e.salary) FROM departments d JOIN employees e ON d.id = e.id GROUP BY d.name;"
success_1, result_1 = execute_sql(attempt_1)
print("Attempt 1 (wrong JOIN column: d.id = e.id):")
print(f"  Success: {success_1}")
print(f"  Result: {result_1}\n")

# Attempt 2: Corrected join column
attempt_2 = "SELECT d.name, AVG(e.salary) AS avg_salary FROM departments d JOIN employees e ON d.id = e.dept_id GROUP BY d.name;"
success_2, result_2 = execute_sql(attempt_2)
print("Attempt 2 (corrected: d.id = e.dept_id):")
print(f"  Success: {success_2}")
print(f"  Result:\n{result_2}")

conn.close()

In [ ]:
def self_correction_loop(
    generate_fn,
    execute_fn,
    question: str,
    max_retries: int = 3
) -> dict[str, object]:
    """
    Generic self-correction loop for text-to-SQL.

    Args:
        generate_fn: Callable that takes (question, error_context) → SQL string
        execute_fn: Callable that takes SQL → (success, result_or_error)
        question: Natural-language question
        max_retries: Maximum correction attempts

    Returns:
        Dict with final SQL, result, attempts, and success flag
    """
    attempts: list[dict[str, str]] = []
    error_context: Optional[str] = None

    for attempt_num in range(1, max_retries + 1):
        sql = generate_fn(question, error_context)
        success, result = execute_fn(sql)

        attempts.append({
            "attempt": attempt_num,
            "sql": sql,
            "success": success,
            "result": result[:200] if isinstance(result, str) else str(result)[:200]
        })

        if success:
            return {"sql": sql, "result": result, "attempts": attempts, "success": True}

        error_context = f"Previous SQL: {sql}\nError: {result}"

    return {"sql": sql, "result": result, "attempts": attempts, "success": False}


print("✅ Self-correction loop defined")
print(f"  Signature: self_correction_loop(generate_fn, execute_fn, question, max_retries=3)")

## 5 · Why narration matters

Raw query results are useful for analysts but **useless for business users**.
Narration transforms a data table into an **actionable insight**:

| Raw result | Narrated insight |
|---|---|
| `Q3_revenue: 1,200,000` | "Revenue dropped 23% in Q3 vs Q2, driven primarily by EMEA region declining 41%." |
| `top_product: Widget-X, count: 4521` | "Widget-X is the best seller with 4,521 units — 2.3× the next product." |

In [ ]:
# Demonstrate: raw table vs narrated insight

sample_data = pd.DataFrame({
    "quarter": ["Q1", "Q2", "Q3", "Q4"],
    "revenue": [1_500_000, 1_560_000, 1_200_000, 1_350_000],
    "region": ["Global", "Global", "Global", "Global"]
})

print("Raw query result:")
print(sample_data.to_string(index=False))
print()


def narrate_revenue_trend(df: pd.DataFrame) -> str:
    """Generate a plain-English narrative from quarterly revenue data."""
    insights: list[str] = []

    # Find biggest quarter-over-quarter change
    df = df.copy()
    df["pct_change"] = df["revenue"].pct_change() * 100

    max_drop_idx = df["pct_change"].idxmin()
    if pd.notna(df.loc[max_drop_idx, "pct_change"]):
        q = df.loc[max_drop_idx, "quarter"]
        prev_q = df.loc[max_drop_idx - 1, "quarter"]
        change = df.loc[max_drop_idx, "pct_change"]
        insights.append(
            f"Revenue dropped {abs(change):.0f}% in {q} vs {prev_q}."
        )

    # Overall trend
    total = df["revenue"].sum()
    insights.append(f"Full-year revenue: ${total:,.0f}.")

    # Best quarter
    best_idx = df["revenue"].idxmax()
    insights.append(
        f"Strongest quarter: {df.loc[best_idx, 'quarter']} "
        f"(${df.loc[best_idx, 'revenue']:,.0f})."
    )

    return " ".join(insights)


narration = narrate_revenue_trend(sample_data)
print("Narrated insight:")
print(f"  {narration}")

## 6 · Schema understanding

The model can only generate correct SQL if it **understands the schema deeply**:

- **Table purposes** — what each table represents in the business domain
- **Column semantics** — what each column means (not just its type)
- **Relationships** — foreign keys, join paths between tables
- **Business glossary** — domain-specific definitions
- **Example queries** — known-good patterns for common questions

In [ ]:
# Build a structured schema documentation format

SCHEMA_DOC: dict[str, dict] = {
    "tables": {
        "customers": {
            "description": "Registered customers with profile and location data",
            "columns": {
                "id": {"type": "INTEGER", "description": "Primary key, auto-increment"},
                "name": {"type": "TEXT", "description": "Full name of the customer"},
                "email": {"type": "TEXT", "description": "Unique email address"},
                "region": {"type": "TEXT", "description": "Geographic region: NA, EMEA, APAC, LATAM"},
                "created_at": {"type": "DATE", "description": "Account creation date"}
            }
        },
        "orders": {
            "description": "Customer purchase orders",
            "columns": {
                "id": {"type": "INTEGER", "description": "Primary key"},
                "customer_id": {"type": "INTEGER", "description": "FK → customers.id"},
                "order_date": {"type": "DATE", "description": "Date the order was placed"},
                "status": {"type": "TEXT", "description": "Order status: pending, shipped, delivered, returned"}
            }
        },
        "order_items": {
            "description": "Line items within an order",
            "columns": {
                "id": {"type": "INTEGER", "description": "Primary key"},
                "order_id": {"type": "INTEGER", "description": "FK → orders.id"},
                "product_id": {"type": "INTEGER", "description": "FK → products.id"},
                "quantity": {"type": "INTEGER", "description": "Number of units ordered"},
                "unit_price": {"type": "REAL", "description": "Price per unit at time of order"}
            }
        }
    },
    "relationships": [
        {"from": "orders.customer_id", "to": "customers.id", "type": "many-to-one"},
        {"from": "order_items.order_id", "to": "orders.id", "type": "many-to-one"},
        {"from": "order_items.product_id", "to": "products.id", "type": "many-to-one"}
    ],
    "business_glossary": {
        "revenue": "SUM(order_items.quantity * order_items.unit_price)",
        "active_customer": "Customer with at least 1 order in last 90 days",
        "churn": "Customer with no orders in last 180 days"
    }
}


def format_schema_for_prompt(schema: dict) -> str:
    """Format schema documentation as context for an LLM prompt."""
    lines: list[str] = ["DATABASE SCHEMA\n"]

    for table_name, table_info in schema["tables"].items():
        lines.append(f"TABLE: {table_name}")
        lines.append(f"  Purpose: {table_info['description']}")
        lines.append("  Columns:")
        for col_name, col_info in table_info["columns"].items():
            lines.append(f"    - {col_name} ({col_info['type']}): {col_info['description']}")
        lines.append("")

    lines.append("RELATIONSHIPS:")
    for rel in schema["relationships"]:
        lines.append(f"  {rel['from']} → {rel['to']} ({rel['type']})")

    lines.append("\nBUSINESS GLOSSARY:")
    for term, definition in schema["business_glossary"].items():
        lines.append(f"  {term}: {definition}")

    return "\n".join(lines)


prompt_context = format_schema_for_prompt(SCHEMA_DOC)
print(prompt_context)

## 7 · Market analysis

The business intelligence (BI) market context:

| Metric | Value |
|---|---|
| **BI market size** | $30B+ and growing ~10% CAGR |
| **SQL-literate users** | ~10% of business users |
| **Analyst backlog** | Average 2–4 week wait for ad-hoc queries |
| **ThoughtSpot ARR** | $250M+ (NL-to-analytics pioneer) |
| **Cost per analyst** | $80–150K/year fully loaded |

**The opportunity**: 90% of data questions never get asked because users
can't write SQL and analysts are bottlenecked. Text-to-SQL removes this barrier.

In [ ]:
# Visualize the market opportunity

categories = ["Can write SQL", "Need analyst help", "Questions never asked"]
percentages = [10, 30, 60]
colors = ["#2ecc71", "#f39c12", "#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(percentages, labels=categories, colors=colors,
            autopct="%1.0f%%", startangle=90)
axes[0].set_title("Business users vs SQL capability")

# Cost comparison
analyst_queries_per_day = 8
ai_queries_per_day = 500
analyst_cost_per_query = 150_000 / (250 * analyst_queries_per_day)  # $75/query
ai_cost_per_query = 0.05  # estimated API cost

labels = ["Human analyst", "AI data analyst"]
costs = [analyst_cost_per_query, ai_cost_per_query]
bars = axes[1].bar(labels, costs, color=["#e74c3c", "#2ecc71"])
axes[1].set_ylabel("Cost per query ($)")
axes[1].set_title("Cost per query comparison")
for bar, cost in zip(bars, costs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 f"${cost:.2f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()
print(f"Cost reduction: {analyst_cost_per_query / ai_cost_per_query:.0f}× cheaper per query")

## 🏋️ Exercises

1. **Ambiguity catalog**: Write 5 more ambiguous business questions and list 3 valid
   SQL interpretations for each. Which interpretation would a business user most likely mean?

2. **Schema documentation**: Pick a database you've worked with (or a public one like Chinook)
   and create a full `SCHEMA_DOC` dictionary with table descriptions, column semantics,
   relationships, and a business glossary of at least 10 terms.

3. **Narration patterns**: Extend `narrate_revenue_trend` to handle (a) year-over-year
   comparisons, (b) top/bottom performers, and (c) anomaly detection (e.g., "unusually high").

## 🎯 Takeaways

- Text-to-SQL translates natural language to executable SQL grounded in schema context
- Ambiguity is the core challenge — one question can map to many valid queries
- Self-correction loops (generate → execute → validate → retry) dramatically improve accuracy
- Narration transforms raw data into actionable business insights
- Schema documentation is the foundation — the model needs deep context to generate correct SQL
- The market is massive: $30B+ BI, 90% of users can't write SQL

## ⏭️ What's next

In **01_architecture.ipynb**, we'll design the complete system architecture:
schema indexing with RAG, question disambiguation, SQL generation with
self-correction, and insight narration — all wired together.

## 📚 References

1. Rajkumar, N. et al. (2022). "Evaluating the Text-to-SQL Capabilities of Large Language Models." arXiv:2204.00498
2. Yu, T. et al. (2018). "Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task." EMNLP.
3. Pourreza, M. & Rafiei, D. (2023). "DIN-SQL: Decomposed In-Context Learning of Text-to-SQL." arXiv:2304.11015
4. ThoughtSpot. "The Rise of AI-Powered Analytics." https://www.thoughtspot.com